In [1]:
import re
import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple
 
import networkx as nx
import torch
from torch_geometric.data import Data
import numpy as np

Rect = Tuple[float, float, float, float]

_DEF_KEYWORDS = frozenset({
    "NEW", "ROUTED", "FIXED", "COVER", "SHAPE", "MASK", "RECT",
    "COREWIRE", "IOWIRE", "BLOCKAGEWIRE", "FILLWIRE",
})

@dataclass
class LEFLayer:
    name:       str
    direction:  str
    pitch_x:    float         
    pitch_y:    float          
    width:      float          
    height:     float = 0.0  
    offset_x:   float = 0.0
    offset_y:   float = 0.0  
    thickness:  float = 0.0     

@dataclass
class Rect:
    xl: float
    yl: float
    xh: float
    yh: float
 
    @property
    def center(self) -> Tuple[float, float]:
        return ((self.xl + self.xh) / 2, (self.yl + self.yh) / 2)
 
    def __repr__(self):
        return f"({self.xl},{self.yl})→({self.xh},{self.yh})"
    
@dataclass
class MacroPin:
    name:        str
    direction:   str = ""
    use:         str = ""
    layer_rects: Dict[str, List[Rect]] = field(default_factory=dict)
 
    def is_supply(self) -> bool: return self.use in ("POWER", "GROUND")
 
    def get_rects(self, layer: Optional[str] = None) -> List[Rect]:
        if layer:
            return self.layer_rects.get(layer, [])
        return [r for rects in self.layer_rects.values() for r in rects]


@dataclass
class LEFMacro:
    name:     str
    cls:      str   = ""     
    size_x:   float = 0.0  
    size_y:   float = 0.0   
    site:     str   = ""
    pins:     Dict[str, MacroPin] = field(default_factory=dict)

    def supply_pins(self) -> List[MacroPin]:
        return [p for p in self.pins.values() if p.is_supply()]

class LEFParser:
    def __init__(self, lef_path: str):
        self.lef_path   = Path(lef_path)
        self.layers:  Dict[str, LEFLayer]  = {}
        self.macros:  Dict[str, LEFMacro]  = {}
 
    def parse(self) -> "LEFParser":
        if not self.lef_path.exists():
            warnings.warn(f"LEF not exist: {self.lef_path}")
            return {}
        text = self.lef_path.read_text(encoding="utf-8", errors="ignore") 
        self._parse_layers(text)
        self._parse_macros(text)
        return self.layers, self.macros
    
    def _parse_layers(self, text: str) -> None:
        for m in re.finditer(r"^LAYER\s+(\w+)\s*\n(.*?)^END\s+\1", text, re.M | re.S):
            name, body = m.group(1), m.group(2)
            layer = LEFLayer(name, direction="", pitch_x=0.0, pitch_y=0.0, width=0.0)
            layer.direction = self._get(body, r"DIRECTION\s+(\w+)")
            layer.width     = self._float(body, r"(?<![A-Z])WIDTH\s+([\d.]+)")
            layer.thickness = self._float(body, r"THICKNESS\s+([\d.]+)")
            layer.height    = self._float(body, r"HEIGHT\s+([\d.]+)")
            pm = re.search(r"PITCH\s+([\d.]+)(?:\s+([\d.]+))?", body)
            if pm:
                layer.pitch_x = float(pm.group(1))
                layer.pitch_y = float(pm.group(2)) if pm.group(2) else layer.pitch_x
            om = re.search(r"OFFSET\s+([\d.\-]+)(?:\s+([\d.\-]+))?", body)
            if om:
                layer.offset_x = float(om.group(1))
                layer.offset_y = float(om.group(2)) if om.group(2) else layer.offset_x
            self.layers[name] = layer
    
    def _parse_macros(self, text: str) -> None:
        for m in re.finditer(r"^MACRO\s+(\w+)\s*\n(.*?)^END\s+\1", text, re.M | re.S):
            name, body = m.group(1), m.group(2)
            macro = LEFMacro(name=name, cls=self._get(body, r"CLASS\s+(\w+(?:\s+\w+)?)\s*;"))
            sm = re.search(r"SIZE\s+([\d.]+)\s+BY\s+([\d.]+)", body)
            if sm:
                macro.size_x, macro.size_y = float(sm.group(1)), float(sm.group(2))
            for pm in re.finditer(r"^\s*PIN\s+(\w+)\s*\n(.*?)^\s*END\s+\1", body, re.M | re.S):
                pin = MacroPin(
                    name      = pm.group(1),
                    direction = self._get(pm.group(2), r"DIRECTION\s+(\w+)"),
                    use       = self._get(pm.group(2), r"USE\s+(\w+)"),
                )
                self._parse_rects(pm.group(2), pin.layer_rects)
                macro.pins[pin.name] = pin
            self.macros[name] = macro
 
    def _parse_rects(self, body: str, store: Dict[str, List[Rect]]) -> None:
        cur = None
        for line in body.splitlines():
            line = line.strip().rstrip(";")
            lm = re.match(r"LAYER\s+(\w+)", line)
            if lm: cur = lm.group(1); continue
            rm = re.match(r"RECT\s+([\d.\-]+)\s+([\d.\-]+)\s+([\d.\-]+)\s+([\d.\-]+)", line)
            if rm and cur:
                store.setdefault(cur, []).append(Rect(*map(float, rm.groups())))
 
    @staticmethod
    def _get(text: str, pat: str, default: str = "") -> str:
        m = re.search(pat, text)
        return m.group(1).strip() if m else default
 
    @staticmethod
    def _float(text: str, pat: str) -> float:
        m = re.search(pat, text)
        return float(m.group(1)) if m else 0.0


@dataclass
class ViaRule:
    name: str
    bot_layer: str
    cut_layer: str
    top_layer: str
    resistance: float    

@dataclass
class PDNSegment:
    net_name: str
    layer: str
    x1: float
    y1: float
    x2: float
    y2: float
    width: float

@dataclass
class PDNVia:
    net_name: str
    via_name: str
    x: float
    y: float
    bot_layer: str = ""
    top_layer: str = ""
    num_cuts: int  = 1
    cut_area: float = 0.0

class DEFParser:
    def __init__(self, def_path: str, power_nets: List[str], dbu: int = 2000):
        self.def_path   = Path(def_path)
        self.power_nets = set(power_nets)
        self.dbu        = dbu
        self.segments:  List[PDNSegment] = []
        self.vias:      List[PDNVia]     = []
        self.die_area:  Tuple[float, float, float, float] = (0, 0, 0, 0)
        self.via_layer_map: Dict[str, Tuple[str, str]] = {}
        self.via_info:  Dict[str, Tuple[str, str, int, float]] = {}

    def parse(self) -> Tuple[List[PDNSegment], List[PDNVia]]:
        if not self.def_path.exists():
            warnings.warn(f"DEF not exist: {self.def_path}")
            return [], []
        text = self.def_path.read_text(encoding="utf-8", errors="ignore")
        self._parse_units(text)
        self._parse_die_area(text)
        self._parse_via_defs(text)
        self._parse_specialnets(text)
        return self.segments, self.vias
    
    def _parse_units(self, text: str) -> None:
        m = re.search(r"UNITS\s+DISTANCE\s+MICRONS\s+(\d+)", text)
        if m:
            self.dbu = int(m.group(1))
    
    def _parse_die_area(self, text: str) -> None:
        m = re.search(r"DIEAREA\s*\(\s*([\d\-]+)\s+([\d\-]+)\s*\)\s*\(\s*([\d\-]+)\s+([\d\-]+)\s*\)", text)
        if m:
            self.die_area = tuple(int(v) / self.dbu for v in m.groups())

    def _parse_via_defs(self, text: str) -> None:
        m = re.search(r"\bVIAS\b\s+\d+\s*;(.*?)END\s+VIAS", text, re.S)
        if not m:
            return
        current: dict = {}

        def _flush(name, d):
            la = d.get("la", "")
            lb = d.get("lb", "")
            try:
                lo, hi = sorted([la, lb], key=lambda s: int(re.search(r'\d+', s).group()))
            except Exception:
                lo, hi = la, lb
            num_cuts = d.get("rows", 1) * d.get("cols", 1)
            cut_w    = d.get("cut_w", 0.0) / self.dbu
            cut_h    = d.get("cut_h", 0.0) / self.dbu
            self.via_info[name] = (lo, hi, num_cuts, cut_w * cut_h)

        current_name = None
        for line in m.group(1).splitlines():
            name_m = re.match(r"\s*-\s+(\S+)", line)
            if name_m:
                if current_name:
                    _flush(current_name, current)
                current_name = name_m.group(1)
                current = {}
                continue
            if not current_name:
                continue
   
            layers_m = re.search(r"\+\s+LAYERS\s+(\w+)\s+\w+\s+(\w+)", line)
            if layers_m:
                current["la"], current["lb"] = layers_m.group(1), layers_m.group(2)
            cut_m = re.search(r"\+\s+CUTSIZE\s+([\d.]+)\s+([\d.]+)", line)
            if cut_m:
                current["cut_w"] = float(cut_m.group(1))
                current["cut_h"] = float(cut_m.group(2))
            rc_m = re.search(r"\+\s+ROWCOL\s+(\d+)\s+(\d+)", line)
            if rc_m:
                current["rows"] = int(rc_m.group(1))
                current["cols"] = int(rc_m.group(2))
        if current_name:
            _flush(current_name, current)

    def _parse_specialnets(self, text: str) -> None:
        m = re.search(r"SPECIALNETS\s+\d+\s*;(.*?)END\s+SPECIALNETS", text, re.S)
        if not m:
            return
        block = m.group(1)
        for net_block in re.split(r"\n\s*-\s+", block)[1:]:
            net_name_m = re.match(r"(\w+)", net_block.strip())
            if not net_name_m:
                continue
            net_name = net_name_m.group(1)
            if net_name not in self.power_nets:
                continue
            self._parse_net_block(net_name, net_block)
 
    def _parse_net_block(self, net_name: str, block: str) -> None:
        current_layer = None
        current_width = 0.0
        for sub in re.split(r"\+\s*NEW\s+", block):
            layer_m = re.search(r"(?:ROUTED|FIXED|COVER)\s+(\w+)\s+([\d.]+)", sub)
            if layer_m:
                current_layer = layer_m.group(1)
                current_width = float(layer_m.group(2)) / self.dbu
            if current_layer is None:
                continue

            coords_raw = re.findall(r"\(\s*([\d\-*]+)\s+([\d\-*]+)\s*\)", sub)
            via_names = [
                w for w in re.findall(r"\)\s*([A-Za-z]\w*)", sub)
                if w.upper() not in _DEF_KEYWORDS
            ]

            prev = None
            for i, (sx, sy) in enumerate(coords_raw):
                x = prev[0] if sx == '*' else int(sx) / self.dbu
                y = prev[1] if sy == '*' else int(sy) / self.dbu
                if prev is not None and current_width > 0:
                    px, py = prev
                    if not (x == px and y == py):
                        self.segments.append(PDNSegment(
                            net_name=net_name, layer=current_layer,
                            x1=px, y1=py, x2=x, y2=y, width=current_width
                        ))
                if i < len(via_names):
                    vname = via_names[i]
                    info  = self.via_info.get(vname)
                    bot   = info[0] if info else ""
                    top   = info[1] if info else ""
                    self.vias.append(PDNVia(
                        net_name=net_name, via_name=vname, x=x, y=y,
                        bot_layer=bot, top_layer=top,
                        num_cuts=info[2] if info else 1,
                        cut_area=info[3] if info else 0.0,
                    ))
                prev = (x, y)

_RC_28NM: Dict[str, Tuple[float, float]] = {
    "M1": (0.130, 0.214),  
    "M2": (0.110, 0.188),
    "M3": (0.095, 0.166),
    "M4": (0.082, 0.153),
    "M5": (0.073, 0.137),
    "M6": (0.065, 0.128),
    "M7": (0.045, 0.044),  
    "M8": (0.005, 0.005),   
    "AP": (0.022, 0.022),   
}

_RHO_CU = 1.72e-8
_RHO_AL = 2.65e-8

_RHO_VIA_28NM: Dict[str, float] = {
    "M1": 2.0,   # VIA1: M1-M2
    "M2": 1.5,   # VIA2: M2-M3
    "M3": 1.2,   # VIA3: M3-M4
    "M4": 1.0,   # VIA4: M4-M5
    "M5": 0.8,   # VIA5: M5-M6
    "M6": 0.6,   # VIA6: M6-M7
    "M7": 0.5,   # VIA7: M7-M8
    "M8": 4.0,   # RV:   M8-AP
}
_RHO_VIA_DEFAULT = 1.5  
 


def _layer_rc(layer) -> Tuple[float, float]:
    if layer.thickness > 0:
        rho = _RHO_AL if layer.name == "AP" else _RHO_CU
        r   = rho / (layer.thickness * 1e-6)   
        c   = _RC_28NM.get(layer.name, (0, 0.005))[1]
        return r, c
    return _RC_28NM.get(layer.name, (0.08, 0.020))

def _via_resistance(bot_layer: str, num_cuts: int, cut_area: float) -> float:
    if num_cuts <= 0 or cut_area <= 0:
        return 10.0
    rho_c = _RHO_VIA_28NM.get(bot_layer, _RHO_VIA_DEFAULT)
    return rho_c / (num_cuts * cut_area)


class PDNGraphBuilder:
    TILE_SIZE = 2.25
    NODE_FEAT_NAMES = ["norm_x", "norm_y", "layer_idx", "local_cap", "power"]
    
    def __init__(
        self,
        lef_path: str,
        def_path: str,
        power_path: Optional[str],
        power_nets: List[str],
        target_net: str,
        m1_layer: str = 'M1',
        grid_snap: float = 0.001,  
    ):
        self.target_net = target_net
        self.m1_layer   = m1_layer
        self.grid_snap  = grid_snap

        lef_parser = LEFParser(lef_path)
        self.layers, self.macros = lef_parser.parse()

        routing = [
            n for n, l in self.layers.items()
            if l.direction in ("HORIZONTAL", "VERTICAL")
        ]
        self._layer_order = {name: i for i, name in enumerate(sorted(
            routing,
            key=lambda s: int(re.search(r'\d+', s).group()) if re.search(r'\d+', s) else 0
        ))}

        self._tile_layer_idx = len(self._layer_order)

        def_parser = DEFParser(def_path, power_nets)
        self.segments, self.vias = def_parser.parse()
        self.die_area = def_parser.die_area
        self.power = np.load(power_path) if power_path else None

    def build(self) -> nx.Graph:
        G = nx.Graph()
        self._add_segment_edges(G)
        self._add_via_edges(G)
        if self.power is not None:
            self._add_power_tiles(G, self.power)
        self._compute_node_features(G)
        return G
    
    def to_pyg(self, G: nx.Graph) -> Data:
        nodes = list(G.nodes())
        node_idx = {n: i for i, n in enumerate(nodes)}
        N = len(nodes)

        feat_dim = len(self.NODE_FEAT_NAMES)
        x   = torch.zeros(N, feat_dim, dtype=torch.float32)
        pos = torch.zeros(N, 2,        dtype=torch.float32)
        
        T = 0
        for node in nodes:
            p = G.nodes[node].get("power")
            if p is not None:
                T = len(p)
                break
        power_out = torch.zeros(N, max(T, 1), dtype=torch.float32)

        for i, node in enumerate(nodes):
            attrs = G.nodes[node]
            x[i] = torch.tensor([
                attrs.get("norm_x",        0.0),
                attrs.get("norm_y",        0.0),
                attrs.get("layer_idx",     0.0),
                attrs.get("local_cap",     0.0),
                0.0,
            ], dtype=torch.float32)
            pos[i] = torch.tensor(
                [attrs.get("x", 0.0), attrs.get("y", 0.0)], dtype=torch.float32
            )
            p = attrs.get("power")
            if p is not None:
                power_out[i] = torch.tensor(p, dtype=torch.float32)

        src_list, dst_list, edge_feat_list = [], [], []
        for u, v, edata in G.edges(data=True):
            i_u, i_v = node_idx[u], node_idx[v]
            feat = [
                edata.get("resistance",  1.0),
                edata.get("capacitance", 0.0),
                edata.get("layer_diff",  0.0),
            ]
            src_list      += [i_u, i_v]
            dst_list      += [i_v, i_u]
            edge_feat_list += [feat, feat]

        edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
        edge_attr  = torch.tensor(edge_feat_list,       dtype=torch.float32)
 
        return Data(
            x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos,
            power=power_out,
            y=torch.zeros(N, dtype=torch.float32),
        )

    def _add_segment_edges(self, G: nx.Graph) -> None:
        for seg in (s for s in self.segments if s.net_name == self.target_net):
            layer_rule = self.layers.get(seg.layer)
            if layer_rule is None:
                continue
            r_sq, c_um2 = _layer_rc(layer_rule)
            n1 = self._snap_node(seg.x1, seg.y1, seg.layer)
            n2 = self._snap_node(seg.x2, seg.y2, seg.layer)
            length = math.hypot(seg.x2 - seg.x1, seg.y2 - seg.y1)
            width  = seg.width if seg.width > 0 else layer_rule.width
            R = (r_sq * length / width) if width > 0 else 1e3
            C = c_um2 * length * width
            for node, (cx, cy) in [(n1, (seg.x1, seg.y1)), (n2, (seg.x2, seg.y2))]:
                if node not in G:
                    G.add_node(node, x=cx, y=cy,
                               layer_idx=self._layer_order.get(seg.layer, 0),
                               local_cap=0.0)
                G.nodes[node]["local_cap"] = G.nodes[node].get("local_cap", 0.0) + C / 2
            if G.has_edge(n1, n2):
                R_old = G[n1][n2]["resistance"]
                G[n1][n2]["resistance"]  = (R_old * R) / (R_old + R) if R_old + R > 0 else R
                G[n1][n2]["capacitance"] += C
            else:
                G.add_edge(n1, n2, resistance=R, capacitance=C,
                           layer=seg.layer, layer_diff=0.0, edge_type="wire")

    def _add_via_edges(self, G: nx.Graph) -> None:
        for via in (v for v in self.vias if v.net_name == self.target_net):
            if not via.bot_layer or not via.top_layer:
                continue
            n_bot = self._snap_node(via.x, via.y, via.bot_layer)
            n_top = self._snap_node(via.x, via.y, via.top_layer)

            if not G.has_node(n_bot):
                G.add_node(n_bot, x=via.x, y=via.y,
                           layer_idx=self._layer_order.get(via.bot_layer, 0),
                           local_cap=0.0)
            if not G.has_node(n_top):
                G.add_node(n_top, x=via.x, y=via.y,
                           layer_idx=self._layer_order.get(via.top_layer, 0),
                           local_cap=0.0)

            R = _via_resistance(via.bot_layer, via.num_cuts, via.cut_area)
            layer_diff = float(abs(
                self._layer_order.get(via.top_layer, 0) - self._layer_order.get(via.bot_layer, 0)
            ))
            if G.has_edge(n_bot, n_top):
                R_old = G[n_bot][n_top]["resistance"]
                G[n_bot][n_top]["resistance"] = (R_old * R) / (R_old + R) if R_old + R > 0 else R
            else:
                G.add_edge(n_bot, n_top, resistance=R, capacitance=0.0,
                           layer_diff=layer_diff, edge_type="via")
               

    def _add_power_tiles(self, G: nx.Graph, power: np.ndarray) -> None:
        T, ny, nx      = power.shape
        xl, yl, xh, yh = self.die_area
        ts             = self.TILE_SIZE
        m1_idx         = self._layer_order.get(self.m1_layer, 0) 
        _R_INJECT = 0.130 * 0.075 / 0.05

        inject_by_tile = {}
        for node, attrs in G.nodes(data=True):
            if attrs.get("layer_idx") != m1_idx:
                continue
            col = max(0, min(int((attrs["x"] - xl) / ts), nx - 1))
            row = max(0, min(int((attrs["y"] - yl) / ts), ny - 1))
            inject_by_tile.setdefault((col, row), []).append(node)

        for row in range(ny):
            for col in range(nx):
                tile_id = ("tile", col, row)
                G.add_node(tile_id,
                           x=xl + (col + 0.5) * ts,
                           y=yl + (row + 0.5) * ts,
                           layer_idx=self._tile_layer_idx,
                           power=power[:, row, col],
                           local_cap=0.0)
                for m1_node in inject_by_tile.get((col, row), []):
                    G.add_edge(tile_id, m1_node,
                               resistance=_R_INJECT, capacitance=0.0,
                               layer_diff=1.0, edge_type="inject")

    def _compute_node_features(self, G: nx.Graph) -> None:
        x0, y0, x1, y1 = self.die_area
        w    = x1 - x0 if x1 > x0 else 1.0
        h    = y1 - y0 if y1 > y0 else 1.0
        max_cap = max((G.nodes[n].get("local_cap", 0.0) for n in G.nodes()), default=1.0) or 1.0
        n_layers = len(self._layer_order) + 1  
        for node in G.nodes():
            attrs = G.nodes[node]
            nx_ = attrs.get("x", 0.0)
            ny_ = attrs.get("y", 0.0)
            attrs["norm_x"]         = (nx_ - x0) / w
            attrs["norm_y"]         = (ny_ - y0) / h
            attrs["layer_idx"]      = attrs.get("layer_idx", 0) / n_layers
            attrs["local_cap"]      = attrs.get("local_cap", 0.0) / max_cap
            attrs["power"]          = attrs.get("power") 

    def _snap_node(self, x: float, y: float, layer: str) -> Tuple:
        s = self.grid_snap
        return (round(x / s) * s, round(y / s) * s, layer)


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops
from torch import Tensor
from typing import Tuple

VDD = 0.810

class PDNConv(MessagePassing):
    def __init__(self, in_dim: int, edge_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__(aggr="sum")
 
        self.msg_mlp = nn.Sequential(
            nn.Linear(in_dim * 2 + edge_dim, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(in_dim + out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.norm = nn.LayerNorm(out_dim)

        self.res = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
 
    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor) -> Tensor:
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.update_mlp(torch.cat([x, out], dim=-1))
        return self.norm(out + self.res(x))
 
    def message(self, x_i: Tensor, x_j: Tensor, edge_attr: Tensor) -> Tensor:
        return self.msg_mlp(torch.cat([x_i, x_j, edge_attr], dim=-1))
    

class IRDropGNN(nn.Module):  
    def __init__(self, node_dim=4, edge_dim=3, hidden=64, n_layers=6, dropout=0.1):
        super().__init__()
        self.node_enc = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.convs    = nn.ModuleList([PDNConv(hidden, edge_dim, hidden, dropout) for _ in range(n_layers)])
        self.rlc_head = nn.Sequential(nn.Linear(hidden,32), nn.ReLU(), nn.Linear(32,2), nn.Softplus())
        self.gru      = nn.GRUCell(hidden + 1, hidden)
        self.readout  = nn.Sequential(nn.Linear(hidden,32), nn.ReLU(), nn.Linear(32,1), nn.Softplus())
        self.dropout  = nn.Dropout(dropout)


    def forward(self, x, edge_index, edge_attr, power, tile_mask, dt=1.0):
        N, T = power.shape
        h = self.node_enc(x)
        for conv in self.convs:
            h = conv(h, edge_index, edge_attr)
        h = self.dropout(h)

        rlc      = self.rlc_head(h)
        reff_all = rlc[:, 0]
        ceff_all = rlc[:, 1]

        h_t    = h.clone()
        dv_steps = []
        for t in range(T):
            p_t       = power[:, t] / VDD
            h_t       = self.gru(torch.cat([h, p_t.unsqueeze(-1)], dim=-1), h_t)
            dv_t      = self.readout(h_t).squeeze(-1)
            dv_steps.append(dv_t)

        dv_all  = torch.stack(dv_steps, dim=1)   # (N, T)
        dv_pred = dv_all[tile_mask]               # (M, T)
        return dv_pred, reff_all[tile_mask], ceff_all[tile_mask]
        
class IRDropLoss(nn.Module):
    def __init__(self, lam: float = 0.1):
        super().__init__()
        self.lam = lam
 
    def forward(
        self,
        dv_pred:  Tensor, 
        dv_label: Tensor,   
        power:    Tensor,   
        reff:     Tensor,   
        ceff:     Tensor,   
        dt:       float,
    ) -> Tuple[Tensor, Tensor, Tensor]:
        
        l_data = nn.functional.mse_loss(dv_pred, dv_label)
        I_inj = power / VDD
        I_R = dv_pred / reff.clamp(min=1e-3).unsqueeze(1)
        dv_diff = torch.zeros_like(dv_pred)
        dv_diff[:, 1:] = dv_pred[:, 1:] - dv_pred[:, :-1]
        I_C = ceff.unsqueeze(1) * dv_diff / dt

        residual = I_C + I_R - I_inj
        l_phy = (residual ** 2).mean()
        loss = l_data + self.lam * l_phy
        return loss, l_data, l_phy
    
def train_one_epoch(model, data, label, optimizer, loss_fn, dt, device):
        model.train()
        optimizer.zero_grad()
        x          = data.x[:, :4].to(device)
        edge_index = data.edge_index.to(device)
        edge_attr  = data.edge_attr.to(device)
        power      = data.power.to(device)          
        tile_mask  = (data.power.sum(dim=1) > 0)    
        tile_mask  = tile_mask.to(device)

        dv_pred, reff, ceff = model(x, edge_index, edge_attr, power, tile_mask, dt)
 
        tile_power = power[tile_mask]           
        loss, l_data, l_phy = loss_fn(dv_pred, label.to(device), tile_power, reff, ceff, dt)
    
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    
        return loss.item(), l_data.item(), l_phy.item()

@torch.no_grad()
def evaluate(model, data, label, loss_fn, dt, device):
    model.eval()
 
    x          = data.x[:, :4].to(device)
    edge_index = data.edge_index.to(device)
    edge_attr  = data.edge_attr.to(device)
    power      = data.power.to(device)
    tile_mask  = (data.power.sum(dim=1) > 0).to(device)
 
    dv_pred, reff, ceff = model(x, edge_index, edge_attr, power, tile_mask, dt)
 
    tile_power = power[tile_mask]
    loss, l_data, l_phy = loss_fn(dv_pred, label.to(device), tile_power, reff, ceff, dt)
 

    mae  = (dv_pred - label.to(device)).abs().mean().item()
    pmax = (dv_pred - label.to(device)).abs().max().item()   
 
    return loss.item(), mae, pmax


def build_model_and_optimizer(
    freq_hz:   float,       
    T:         int = 20,
    hidden:    int = 64,
    n_layers:  int = 6,
    lr:        float = 1e-3,
    lam:       float = 0.1,
    device:    str = "cpu",
):
    dt = 1.0 / (freq_hz * T)  
 
    model = IRDropGNN(
        node_dim=4, edge_dim=3,
        hidden=hidden, n_layers=n_layers,
    ).to(device)
 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    loss_fn   = IRDropLoss(lam=lam)
 
    total = sum(p.numel() for p in model.parameters())
    print(f"params: {total:,}")
    print(f"dt = {dt*1e12:.1f} ps  (freq={freq_hz/1e9:.1f} GHz, T={T})")
 
    return model, optimizer, scheduler, loss_fn, dt



In [9]:
import torch
import numpy as np

LEF_PATH   = "circuitnet.lef"
DEF_PATH   = "826-RISCY-a-2-c2-u0.8-m4-p8-f0.def"
POWER_PATH = "data/power_t/826-RISCY-a-2-c2-u0.8-m4-p8-f0"
LABEL_PATH = "data/IR_drop/826-RISCY-a-2-c2-u0.8-m4-p8-f0"
POWER_NETS = ["VDD", "VSS"]
TARGET_NET = "VDD"

builder = PDNGraphBuilder(
    lef_path=LEF_PATH, def_path=DEF_PATH,
    power_path=POWER_PATH, power_nets=POWER_NETS, target_net=TARGET_NET,
)
G    = builder.build()
data = builder.to_pyg(G)
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]//2}")
print(f"power shape: {data.power.shape}")

power_raw = np.load(POWER_PATH )   
ir_raw    = np.load(LABEL_PATH )   
print(f"IR drop label shape: {ir_raw.shape}")

T, ny, nx_g = power_raw.shape

if ir_raw.ndim == 2:                           
    label_full = torch.tensor(
        np.stack([ir_raw] * T, axis=0).reshape(T, -1).T,   
        dtype=torch.float32
    )
elif ir_raw.ndim == 3:                       
    label_full = torch.tensor(
        ir_raw.reshape(T, -1).T,               
        dtype=torch.float32
    )

active_tiles = (power_raw.sum(axis=0).reshape(-1) > 0)    
label = label_full[active_tiles] / 1000.0

tile_mask_global = (data.power.sum(dim=1) > 0)
print(f"active tiles: {tile_mask_global.sum().item()} / {ny*nx_g}")
assert tile_mask_global.sum().item() == label.shape[0], \
    f"label/mask 不匹配: {label.shape[0]} vs {tile_mask_global.sum().item()}"

device  = "cpu"
FREQ_HZ =   5e8
dt_val  = 1.0 / (FREQ_HZ * T)

model     = IRDropGNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn   = IRDropLoss(lam=0.0)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

loss, l_data, l_phy = train_one_epoch(
    model, data, label, optimizer, loss_fn, dt_val, device
)
print(f"train → loss={loss:.4f}  data={l_data:.4f}  phy={l_phy:.4f}")

loss, mae, pmax = evaluate(model, data, label, loss_fn, dt_val, device)
print(f"eval  → MAE={mae*1000:.3f} mV  Peak={pmax*1000:.3f} mV")




nodes=83868, edges=13679
power shape: torch.Size([83868, 20])
IR drop label shape: (263, 264)
active tiles: 22297 / 69432
params: 184,803
train → loss=0.4889  data=0.4889  phy=1848867725246464.0000
eval  → MAE=587.824 mV  Peak=600.901 mV
